In [0]:
%python

from delta.tables import DeltaTable
from pyspark.sql import functions as F

tabela = "blueprint_spark.bronze.clientes_hist"

clientes = spark.createDataFrame([
    (1, "Ana",   "Salvador", True),
    (2, "Bruno", "Recife",   True),
    (3, "Carla", "Curitiba", False),
], ["id", "nome", "cidade", "ativo"])

# Gerando algumas versões para ter histórico
clientes.write.mode("overwrite").saveAsTable(tabela)                       # versão 0

spark.createDataFrame([(4, "Diego", "Recife", True)], clientes.columns) \
     .write.mode("append").saveAsTable(tabela)                             # versão 1

dt = DeltaTable.forName(spark, tabela)
dt.update(F.col("id") == 2, {"cidade": F.lit("Salvador")})                 # versão 2
dt.delete(F.col("ativo") == False)                                         # versão 3

spark.table(tabela).show()

+---+-----+--------+-----+
| id| nome|  cidade|ativo|
+---+-----+--------+-----+
|  2|Bruno|Salvador| true|
|  1|  Ana|Salvador| true|
|  4|Diego|  Recife| true|
+---+-----+--------+-----+



In [0]:
spark.read.option("versionAsOf",0).table(tabela).show()


dt.restoreToVersion(0)
spark.table(tabela).show()

+---+-----+--------+-----+
| id| nome|  cidade|ativo|
+---+-----+--------+-----+
|  1|  Ana|Salvador| true|
|  2|Bruno|  Recife| true|
|  3|Carla|Curitiba|false|
+---+-----+--------+-----+

+---+-----+--------+-----+
| id| nome|  cidade|ativo|
+---+-----+--------+-----+
|  1|  Ana|Salvador| true|
|  2|Bruno|  Recife| true|
|  3|Carla|Curitiba|false|
+---+-----+--------+-----+



In [0]:
dt.history().show()

+-------+-------------------+----------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|          userId|            userName|           operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+----------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      6|2026-11-01 23:07:49|4348458721156606|walterbiel1237@gm...|             RESTORE|{version -> 0, ti...|NULL|{1957532596960695}|   96f2152c-2b61-44f...|1101-2141

In [0]:
dt.vacuum(24)

In [0]:
dt.history().show()

+-------+-------------------+----------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|          userId|            userName|           operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+----------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|     10|2026-11-01 23:15:00|4348458721156606|walterbiel1237@gm...|          VACUUM END|{status -> COMPLE...|NULL|{1957532596960695}|   551c53ac-3fc0-47d...|1101-2141

In [0]:
tabela_opt = "blueprint_spark.bronze.vendas_opt"

# Simulando o problema: 10 appends pequenos = muitos arquivos pequenos
spark.range(1000).withColumn("cidade", F.lit("Salvador")).write.mode("overwrite").saveAsTable(tabela_opt)
for i in range(9):
    spark.range(1000).withColumn("cidade", F.lit("Recife")).write.mode("append").saveAsTable(tabela_opt)

In [0]:

dt_opt = DeltaTable.forName(spark, tabela_opt)
dt_opt.detail().select("numFiles").show()

+--------+
|numFiles|
+--------+
|      17|
+--------+



In [0]:
dt_opt.optimize().executeCompaction()

DataFrame[path: string, metrics: struct<autoCompactParallelismStats:void,clusteringMetrics:void,clusteringStats:void,deletionVectorStats:struct<numDeletionVectorRowsRemoved:bigint,numDeletionVectorsRemoved:bigint>,endTimeMs:bigint,fileAgeHistogram:void,filesAdded:struct<avg:double,max:bigint,min:bigint,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<avg:double,max:bigint,min:bigint,totalFiles:bigint,totalSize:bigint>,numBatches:bigint,numBins:bigint,numBytesSkippedToReduceWriteAmplification:bigint,numFilesAdded:bigint,numFilesRemoved:bigint,numFilesSkippedToReduceWriteAmplification:bigint,numFilesSortedWithinFiles:bigint,numTableColumns:bigint,numTableColumnsWithStats:bigint,partitionsOptimized:bigint,preserveInsertionOrder:boolean,recompressionCodec:void,skippedArchivedFiles:bigint,startTimeMs:bigint,totalClusterParallelism:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,totalScheduledTasks:bigint,totalTaskExecutionTimeMs:bigint,zOrderStats:void>]

In [0]:
dt_opt.detail().show()

+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
|format|                  id|                name|description|location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|          properties|minReaderVersion|minWriterVersion|       tableFeatures|          statistics|clusterByAuto|
+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
| delta|b26644be-4522-4d7...|blueprint_spark.b...|       NULL|        |2026-11-01 23:20:...|2026-11-01 23:23:56|              []|               []|       1|    

In [0]:
spark.table(tabela_opt).show()

+---+--------+
| id|  cidade|
+---+--------+
|  0|Salvador|
|  1|Salvador|
|  2|Salvador|
|  3|Salvador|
|  4|Salvador|
|  5|Salvador|
|  6|Salvador|
|  7|Salvador|
|  8|Salvador|
|  9|Salvador|
| 10|Salvador|
| 11|Salvador|
| 12|Salvador|
| 13|Salvador|
| 14|Salvador|
| 15|Salvador|
| 16|Salvador|
| 17|Salvador|
| 18|Salvador|
| 19|Salvador|
+---+--------+
only showing top 20 rows


In [0]:
dt_opt.optimize().executeZOrderBy("cidade")


DataFrame[path: string, metrics: struct<autoCompactParallelismStats:void,clusteringMetrics:void,clusteringStats:void,deletionVectorStats:struct<numDeletionVectorRowsRemoved:bigint,numDeletionVectorsRemoved:bigint>,endTimeMs:bigint,fileAgeHistogram:void,filesAdded:struct<avg:double,max:void,min:void,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<avg:double,max:void,min:void,totalFiles:bigint,totalSize:bigint>,numBatches:bigint,numBins:bigint,numBytesSkippedToReduceWriteAmplification:bigint,numFilesAdded:bigint,numFilesRemoved:bigint,numFilesSkippedToReduceWriteAmplification:bigint,numFilesSortedWithinFiles:bigint,numTableColumns:bigint,numTableColumnsWithStats:bigint,partitionsOptimized:bigint,preserveInsertionOrder:boolean,recompressionCodec:void,skippedArchivedFiles:bigint,startTimeMs:bigint,totalClusterParallelism:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,totalScheduledTasks:bigint,totalTaskExecutionTimeMs:bigint,zOrderStats:struct<inputCubeFiles:struct<num